# IMPLEMENTING SELF ATTENTION WITH TRAINABLE WEIGHTS

## Step 0 - Setup & Input Embeddings

First we import PyTorch and define our toy sentence **"Today i am learning deepseek course"**. Every token is already represented by a **3-dimensional embedding vector**.

The `inputs` tensor has shape `(6, 3)` — **6 tokens**, each a **3-dim** vector `x^(i)`:

| Token | Vector |
|-------|--------|
| Today    | `[0.43, 0.15, 0.89]` |
| i        | `[0.55, 0.87, 0.66]` |
| am       | `[0.57, 0.85, 0.64]` |
| learning | `[0.22, 0.58, 0.33]` |
| deepseek | `[0.77, 0.25, 0.10]` |
| course   | `[0.05, 0.80, 0.55]` |

These embeddings are the **input** to the self-attention layer. Its job is to turn each vector into a **context vector** that also captures information from the *other* tokens.

In [7]:
import torch
import torch.nn as nn

In [8]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Today     (x^1)
    [0.55, 0.87, 0.66], #  i         (x^2)
    [0.57, 0.85, 0.64], # am         (x^3)
    [0.22, 0.58, 0.33], # learning   (x^4)
    [0.77, 0.25, 0.10], # deepseek   (x^5)
    [0.05, 0.80, 0.55] # course      (x^6)
    ]
)

## Step 1 — Self-Attention with Trainable Weights (`SelfAttention_v2`)

This version introduces **three trainable weight matrices**, implemented with `nn.Linear`. These matrices are what the model actually *learns* during training.

Each input token `x` is projected into three roles:

| Projection | Layer | Meaning |
|-----------|-------|---------|
| **Query** `q` | `W_query` | "What am I looking for?" |
| **Key** `k`   | `W_key`   | "What do I contain?" |
| **Value** `v` | `W_value` | "What information do I pass on?" |

**The `forward` pass computes:**

1. **Project** inputs → `queries`, `keys`, `values`.
2. **Attention scores** — dot product of every query with every key:
   $$\text{scores} = Q K^\top$$
3. **Attention weights** — scale by $\sqrt{d_k}$ (stabilises gradients), then `softmax` so each row sums to 1:
   $$\alpha = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right)$$
4. **Context vectors** — weighted sum of the value vectors:
   $$Z = \alpha V$$

> **Note:** `bias=qkv_bias` (default `False`) — the Q/K/V projections usually omit a bias term.

In [9]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, X):
        keys = self.W_key(X)
        queries = self.W_query(X)
        values = self.W_value(X)

        atten_scores = queries @ keys.T
        atten_weight = torch.softmax(atten_scores / keys.shape[-1] ** 0.5, dim=1)

        context_vector = atten_weight @ values
        return context_vector

## Step 2 — Run the Layer

We set `torch.manual_seed(789)` so the randomly-initialized weights are reproducible, then build the layer with:

- `d_in = 3` → matches the 3-dim input embeddings.
- `d_out = 6` → the size of each query/key/value (and the output context vector).

Passing `inputs` `(6, 3)` through the layer returns **context vectors** of shape `(6, 6)` — one enriched vector per token. The `grad_fn=<MmBackward0>` in the output confirms the tensor is part of the autograd graph, ready for backpropagation.

In [12]:
torch.manual_seed(789)
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Today     (x^1)
    [0.55, 0.87, 0.66], #  i         (x^2)
    [0.57, 0.85, 0.64], # am         (x^3)
    [0.22, 0.58, 0.33], # learning   (x^4)
    [0.77, 0.25, 0.10], # deepseek   (x^5)
    [0.05, 0.80, 0.55] # course      (x^6)
    ]
)
d_in = 3
d_out = 6

self_attention = SelfAttention_v2(d_in, d_out)
print(self_attention(inputs))


tensor([[-0.2954, -0.0179,  0.2729,  0.1805,  0.1888,  0.0142],
        [-0.2975, -0.0187,  0.2750,  0.1815,  0.1901,  0.0144],
        [-0.2975, -0.0187,  0.2749,  0.1814,  0.1900,  0.0142],
        [-0.2938, -0.0173,  0.2722,  0.1802,  0.1882,  0.0156],
        [-0.2938, -0.0174,  0.2699,  0.1790,  0.1872,  0.0116],
        [-0.2948, -0.0176,  0.2740,  0.1812,  0.1892,  0.0172]],
       grad_fn=<MmBackward0>)
